# Flipkart conversion strategy case using public third-party clickstream
**Not Flipkart user data.** Source: REES46. Notebook does not invent results; obtain the original e-commerce events CSV first. Charts are computed only after execution.

In [ ]:
from pathlib import Path
import pandas as pd,numpy as np,matplotlib.pyplot as plt,hashlib
path=Path('../data/ecommerce_events_raw.csv')
assert path.exists(), 'TO BE COMPLETED: download genuine REES46 behavior-events CSV and place here.'
print('SHA256',hashlib.sha256(path.read_bytes()).hexdigest())
df=pd.read_csv(path)
print('rows, cols:',df.shape); print(df.dtypes);display(df.head(5))

In [ ]:
required=['event_time','event_type','product_id','category_id','category_code','brand','price','user_id','user_session']
assert set(required).issubset(df.columns),f'Missing: {set(required)-set(df.columns)}'
audit=pd.DataFrame({'field':df.columns,'dtype':df.dtypes.astype(str).values,'null_count':df.isna().sum().values,'distinct':df.nunique(dropna=True).values})
display(audit); print('full duplicate rows:',df.duplicated().sum()); print('event types',df.event_type.value_counts(dropna=False).to_dict())
assert set(df.event_type.dropna().unique()).issubset({'view','cart','remove_from_cart','purchase'}),'Unexpected event types - investigate'
assert df.user_session.notna().any(),'No session identifiers; session funnel unsupported'


In [ ]:
events=df.copy();events['event_at']=pd.to_datetime(events.event_time,utc=True,errors='coerce');events['price_num']=pd.to_numeric(events.price,errors='coerce')
print('Invalid timestamps:',events.event_at.isna().sum(),'Session missing:',events.user_session.isna().sum())
# Raw file is never altered. Session metrics exclude rows with missing IDs only, explicitly.
usable=events.loc[events.user_session.notna()].copy()
session=pd.crosstab(usable.user_session.astype(str),usable.event_type).gt(0)
for flag in ['view','cart','remove_from_cart','purchase']:
    if flag not in session.columns:session[flag]=False
session=session[['view','cart','remove_from_cart','purchase']]
session['cart_without_purchase']=session['cart'] & ~session['purchase']
num_cart=int(session['cart'].sum());num_no_purchase=int(session.cart_without_purchase.sum())
metrics={'observed_sessions':len(session),'view_sessions':int(session['view'].sum()),'cart_sessions':num_cart,'purchase_sessions':int(session['purchase'].sum()),'cart_no_purchase_sessions':num_no_purchase,'cart_no_purchase_pct':100*num_no_purchase/num_cart if num_cart else np.nan,'session_purchase_pct':100*session['purchase'].mean()}
print(metrics)
Path('../charts').mkdir(exist_ok=True)
pd.Series(metrics).to_csv('../charts/kpi_actual_after_execution.csv')


In [ ]:
fig,ax=plt.subplots(figsize=(8,4.5)); vals=[metrics['view_sessions'],metrics['cart_sessions'],metrics['purchase_sessions']];ax.bar(['Viewed','Added to cart','Purchased'],vals);ax.set_ylabel('Distinct observed sessions');ax.set_title('Session funnel - public sample (NOT Flipkart)');fig.tight_layout();fig.savefig('../charts/01_session_funnel.png',dpi=175);plt.show()
by_date=usable.groupby([usable.event_at.dt.date,'event_type']).size().unstack(fill_value=0)
fig,ax=plt.subplots(figsize=(10,4));by_date.reindex(columns=['view','cart','purchase'],fill_value=0).plot(ax=ax);ax.set_ylabel('Event count');ax.set_title('Daily event volumes (not sessions)');fig.tight_layout();fig.savefig('../charts/02_daily_events.png',dpi=175);plt.show()
prod=usable.loc[usable.event_type.eq('cart')].groupby('product_id').size().nlargest(12).sort_values()
fig,ax=plt.subplots(figsize=(9,5));prod.plot.barh(ax=ax);ax.set_xlabel('Cart event count');ax.set_title('Top products by cart events');fig.tight_layout();fig.savefig('../charts/03_cart_products.png',dpi=175);plt.show()
cat=usable.loc[usable.event_type.eq('purchase')].groupby('category_code').size().nlargest(12).sort_values()
if not cat.empty:
 fig,ax=plt.subplots(figsize=(9,5));cat.plot.barh(ax=ax);ax.set_xlabel('Purchase events');ax.set_title('Top coded categories - purchase events');fig.tight_layout();fig.savefig('../charts/04_purchased_categories.png',dpi=175);plt.show()


In [ ]:
session.reset_index().to_csv('../python/session_metrics.csv',index=False)
print('Exported session flags for Power BI. Review event censoring and missing sessions before interpretation.')